# Day 59：Capstone Dev Day 1 — 项目骨架与企业财务/合规知识库构建

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段——**Day 59（Capstone 开发第 1 天）**！今天我们正式搭建 Capstone 独立工程骨架（`v1.0.0-day59`），并实现 `CapstoneKBInitializer` 知识库初始化流水线，完成：
> 1. **多格式企业语料加载**：解析带 Frontmatter 的财务报告/合规制度 `.md` 与结构化财务台账 `.csv`；
> 2. **结构化元数据注入**：自动提取并标准化 `department`、`fiscal_year`、`doc_type`；
> 3. **JSON 与 SQLite 双路持久化**：生成可审计的 `kb_index.json` 与支持幂等更新及元数据过滤的 `kb_index.sqlite`。

## 1. 环境初始化与导入 `project_iterations/day_59` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day59_dir = repo_root / "project_iterations" / "day_59"
if str(day59_dir) not in sys.path:
    sys.path.insert(0, str(day59_dir))

from zhishu_agent import CapstoneKBInitializer, IngestionStats, KBChunk

print("已加载模块路径:", day59_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_59


## 2. 生成企业财务/合规样例语料并执行知识库构建与持久化

In [2]:
import json
import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    base = Path(tmp_dir)
    corpus_dir = base / "corpus"
    json_path = base / "artifacts" / "kb_index.json"
    sqlite_path = base / "artifacts" / "kb_index.sqlite"

    created_files = CapstoneKBInitializer.create_sample_corpus(corpus_dir)
    print("生成样例文件:", [p.name for p in created_files])

    initializer = CapstoneKBInitializer(chunk_size=260, chunk_overlap=40, rows_per_csv_chunk=2)
    chunks, stats = initializer.build_and_persist(
        input_dir=corpus_dir,
        json_path=json_path,
        sqlite_path=sqlite_path,
    )

    print(f"处理文件: {stats.processed_files}/{stats.total_files}, 总切片数: {stats.total_chunks}")
    print("部门分布:", stats.departments)
    print("财年分布:", stats.fiscal_years)
    print("文档类型分布:", stats.doc_types)

    assert stats.processed_files == 3
    assert json_path.exists() and sqlite_path.exists()

    json_data = json.loads(json_path.read_text(encoding="utf-8"))
    assert json_data["version"] == "v1.0.0-day59"
    assert json_data["total_chunks"] == len(chunks)

    # 验证 SQLite 幂等重复写入
    initializer.save_sqlite_index(chunks, sqlite_path)
    all_rows = initializer.query_sqlite(sqlite_path, limit=100)
    assert len(all_rows) == len(chunks)

    # 验证按部门 + 财年 + 关键词检索
    comp_hits = initializer.query_sqlite(
        db_path=sqlite_path,
        department="compliance",
        fiscal_year="FY2026",
        keyword="脱敏",
    )
    print("合规制度检索命中数:", len(comp_hits))
    for h in comp_hits:
        print(f"  -> [{h['chunk_id']}] {h['section_heading']} | {h['content'][:45]}...")
    assert len(comp_hits) >= 1

    fin_hits = initializer.query_sqlite(
        db_path=sqlite_path,
        department="finance",
        fiscal_year="FY2025",
    )
    print("FY2025 财务文档命中数:", len(fin_hits))
    assert len(fin_hits) >= 3

生成样例文件: ['fy2025_q4_financial_report.md', 'compliance_expense_policy_2026.md', 'fy2025_dept_budget_ledger.csv']
处理文件: 3/3, 总切片数: 8
部门分布: {'compliance': 2, 'finance': 6}
财年分布: {'FY2026': 2, 'FY2025': 6}
文档类型分布: {'compliance_policy': 2, 'financial_ledger': 4, 'financial_report': 2}
合规制度检索命中数: 1
  -> [1645e243430d90dd448a] 第二章 敏感信息与审计留存规范 | ## 第二章 敏感信息与审计留存规范

所有进入智能体问答系统的财务凭证必须经过 PII ...
FY2025 财务文档命中数: 6
